# 🚀 KaizenStat — Quick Start (Tabular Data)

**Run this in under 2 minutes. No setup required.**

This notebook demonstrates the full KaizenStat pipeline on the Titanic dataset:

| Step | Method | What it does |
|------|--------|--------------|
| 1 | `fit()` | Register dataset, auto-detect task type |
| 2 | `health()` | Data Health Score 0–100 |
| 3 | `validate()` | Leakage + drift + statistical checks |
| 4 | `fix(safe=True)` | Preview then auto-heal data issues |
| 5 | `train()` | Benchmark 5 models, train the best |
| 6 | `debug_model()` | Root-cause failure analysis |
| 7 | `improve()` | Ranked improvement suggestions |
| 8 | `report()` | Terminal summary + HTML export |

---
> **Dataset:** [Titanic — Kaggle](https://www.kaggle.com/competitions/titanic/data) (loaded automatically via public URL)
>
> **Time:** ~2 minutes on Colab CPU

## Step 0 — Install KaizenStat

Run this once per Colab session.

In [ ]:
!pip install kaizenstat -q
print("✅ KaizenStat installed")

## Step 1 — Load Dataset

We load the Titanic training CSV directly from a public GitHub mirror of the Kaggle dataset.
No Kaggle login needed.

In [ ]:
import pandas as pd
from kaizenstat import DataDoctor

# Load Titanic dataset (Kaggle competition data via public mirror)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
df.head()

## Step 2 — Fit the DataDoctor

`fit()` registers the dataset and **auto-detects**:
- Task type (classification vs regression)
- Column types (numeric, categorical, text, datetime)
- Dataset mode (tabular vs NLP)

We're predicting `Survived` (binary classification: 0 = died, 1 = survived).

In [ ]:
doctor = DataDoctor()
doctor.fit(df, target="Survived")

## Step 3 — Data Health Score

`health()` scans the dataset and returns a **0–100 score** covering:
- Missing values (count + percentage per column)
- Duplicate rows
- Outliers (IQR method)
- Class imbalance ratio
- Constant / low-variance features

**Target:** Score > 80 for production-ready data.

In [ ]:
health_result = doctor.health()
print(f"\n📊 Data Health Score: {health_result.score} / 100")

## Step 4 — Validate

`validate()` runs deeper statistical checks:
- **Data leakage** — features with suspiciously high correlation to target
- **Feature drift** — distribution shift between train/test splits (KS test)
- **Multicollinearity** — VIF > 10 pairs
- **Statistical assumptions** — skewness, kurtosis violations

This catches the issues that cause production failures.

In [ ]:
validation_result = doctor.validate()

## Step 5 — Auto-Fix

`fix(safe=True)` shows a **preview of all planned fixes** before applying them:
- Imputes missing values (median for numeric, mode for categorical)
- Drops constant columns
- Removes duplicate rows
- Encodes categorical features

`safe=True` means only low-risk, reversible fixes are applied. Use `safe=False` for aggressive fixes.

In [ ]:
fixed_df = doctor.fix(safe=True)
print(f"\n✅ Fixed dataset shape: {fixed_df.shape}")
print(f"   Missing values remaining: {fixed_df.isnull().sum().sum()}")

## Step 6 — Train (Benchmark + Best Model)

`train()` runs a full **model benchmark** across 5 algorithms:
- Logistic Regression
- Random Forest
- Gradient Boosting
- XGBoost (if installed)
- LightGBM (if installed)

Then trains the winner with cross-validation and returns detailed metrics.

**cv=5** means 5-fold cross-validation. Increase to `cv=10` for more robust estimates.

In [ ]:
train_result = doctor.train(cv=5)

print(f"\n🏆 Best model: {train_result.model_name}")
print(f"   Test score:  {train_result.test_score:.4f}")
print(f"   Train score: {train_result.train_score:.4f}")

## Step 7 — Debug Model

`debug_model()` performs **root-cause failure analysis**:
- **Data vs model blame** — is the failure due to bad data or wrong model?
- **Feature importance** — which features drive predictions
- **Failure slices** — which subgroups the model fails on
- **Train/test gap** — overfit or underfit diagnosis

This is the most unique part of KaizenStat — no other tool does this automatically.

In [ ]:
debug_result = doctor.debug_model()

print(f"\n🔍 Train score: {debug_result.train_score:.4f}")
print(f"   Test score:  {debug_result.test_score:.4f}")
print(f"   Gap:         {debug_result.gap:.4f}")

## Step 8 — Improve

`improve()` generates a **prioritised improvement plan** based on all previous steps:
- HIGH priority fixes (biggest expected gain)
- MEDIUM priority (moderate improvement)
- LOW priority (polish)

Suggestions are data-aware — they account for your specific dataset issues, not generic advice.

In [ ]:
improvement_report = doctor.improve()

## Step 9 — Report

`report()` generates:
1. **Terminal summary** — compact view of all results
2. **HTML report** — self-contained file you can share with your team

In Colab, we display the HTML report inline below.

In [ ]:
report_path = doctor.report(output_path="titanic_report.html")
print(f"\n📄 Report saved to: {report_path}")

In [ ]:
# Display the HTML report inline in Colab
from IPython.display import IFrame, display
display(IFrame(src='titanic_report.html', width='100%', height='600px'))

## Bonus — Pipeline Confidence Score

Get a single 0–100 number summarising your entire pipeline's production readiness.

In [ ]:
confidence = doctor.pipeline_confidence()
print(f"\n🎯 Pipeline Confidence: {confidence} / 100")

---
## Summary

You just ran the full KaizenStat pipeline:

```python
doctor = DataDoctor()
doctor.fit(df, target="Survived")   # auto-detect task type
doctor.health()                      # data health score 0–100
doctor.validate()                    # leakage + drift checks
doctor.fix(safe=True)                # preview then auto-heal
doctor.train()                       # benchmark + train best model
doctor.debug_model()                 # root-cause failure analysis
doctor.improve()                     # ranked improvement suggestions
doctor.report()                      # terminal summary + HTML export
```

### Next Steps
- **Try on your own CSV** — replace the URL with `pd.read_csv('your_data.csv')`
- **Run with `tune=True`** — `doctor.train(tune=True)` for hyperparameter tuning
- **Try the Intermediate notebook** for deeper analysis
- **Install:** `pip install kaizenstat`
- **Docs:** [github.com/masuddarrahaman/KaizenStat-Library](https://github.com/masuddarrahaman/KaizenStat-Library)

---
*Built by [Masuddar Rahaman](https://github.com/masuddarrahaman) · KaizenStat v0.5.1 · MIT License*